# Direct and reciprocal Bravais lattices

**Prerequisite:** [Direct and reciprocal lattice bases](../../../docs/lecturenotes/solidstate/lattices/direct-and-reciprocal/index.md)

This maintained laboratory constructs physical direct-lattice bases and their reciprocal bases in one, two, and three dimensions. It verifies the defining relation $A^{\mathsf T}B=2\pi I$ and visualizes a two-dimensional oblique lattice.

All numerical vectors in this laboratory use one consistent illustrative length unit. Reciprocal vectors consequently use its inverse. The lattice classes store numerical bases rather than selecting a physical unit.

## Learning objectives

1. Distinguish physical direct-lattice bases from finite integer quotient domains.
2. Construct reciprocal bases from direct bases.
3. Verify direct–reciprocal duality and fundamental-region measures.
4. Generate lattice points from integer indices without notebook-local lattice classes.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.periodic.lattice.lattice1d import (
    DirectLattice1D,
    ReciprocalLattice1D,
)
from projectkoios.physkit.periodic.lattice.lattice2d import (
    DirectLattice2D,
    ReciprocalLattice2D,
)
from projectkoios.physkit.periodic.lattice.lattice3d import (
    DirectLattice3D,
    ReciprocalLattice3D,
)

In [ ]:
direct_1d = DirectLattice1D(2.5)
reciprocal_1d = ReciprocalLattice1D.from_direct_lattice(direct_1d)

direct_2d = DirectLattice2D(
    a1=np.array([1.0, 0.0], dtype=np.float64),
    a2=np.array([0.5, np.sqrt(3.0) / 2.0], dtype=np.float64),
)
reciprocal_2d = ReciprocalLattice2D.from_direct_lattice(direct_2d)

direct_3d = DirectLattice3D(
    a1=np.array([2.0, 0.0, 0.0], dtype=np.float64),
    a2=np.array([0.0, 2.0, 0.0], dtype=np.float64),
    a3=np.array([0.0, 0.0, 2.0], dtype=np.float64),
)
reciprocal_3d = ReciprocalLattice3D.from_direct_lattice(direct_3d)

In [ ]:
direct_bases = (
    direct_1d.primitive_basis,
    direct_2d.A,
    direct_3d.primitive_basis_matrix,
)
reciprocal_bases = (
    reciprocal_1d.primitive_basis,
    reciprocal_2d.primitive_basis,
    reciprocal_3d.primitive_basis_matrix,
)

duality_residuals = tuple(
    direct_basis.T @ reciprocal_basis - 2.0 * np.pi * np.eye(direct_basis.shape[0])
    for direct_basis, reciprocal_basis in zip(
        direct_bases,
        reciprocal_bases,
        strict=True,
    )
)

for residual in duality_residuals:
    assert np.allclose(residual, 0.0, rtol=0.0, atol=2e-15)

duality_maximum_residuals = tuple(
    float(np.max(np.abs(residual))) for residual in duality_residuals
)
duality_maximum_residuals

In [ ]:
assert np.isclose(
    direct_1d.measure * reciprocal_1d.measure,
    2.0 * np.pi,
)
assert np.isclose(
    direct_2d.measure * reciprocal_2d.measure,
    (2.0 * np.pi) ** 2,
)
assert np.isclose(
    direct_3d.measure * reciprocal_3d.measure,
    (2.0 * np.pi) ** 3,
)

measure_products = (
    direct_1d.measure * reciprocal_1d.measure,
    direct_2d.measure * reciprocal_2d.measure,
    direct_3d.measure * reciprocal_3d.measure,
)
measure_products

In [ ]:
integer_axis = np.arange(-3, 4, dtype=np.int64)
first_indices, second_indices = np.meshgrid(
    integer_axis,
    integer_axis,
    indexing="ij",
)
indices_2d = np.column_stack((first_indices.ravel(), second_indices.ravel())).astype(
    np.int64
)
direct_points = direct_2d.vector(indices_2d)
reciprocal_points = reciprocal_2d.vector(indices_2d)

figure, axes = plt.subplots(1, 2, figsize=(11, 5))
for axis, points, first, second, title in (
    (
        axes[0],
        direct_points,
        direct_2d.a1,
        direct_2d.a2,
        "direct lattice",
    ),
    (
        axes[1],
        reciprocal_points,
        reciprocal_2d.b1,
        reciprocal_2d.b2,
        "reciprocal lattice",
    ),
):
    axis.scatter(points[:, 0], points[:, 1], s=18, color="black")
    axis.quiver(
        0.0,
        0.0,
        first[0],
        first[1],
        angles="xy",
        scale_units="xy",
        scale=1.0,
        color="tab:red",
        label="first primitive vector",
    )
    axis.quiver(
        0.0,
        0.0,
        second[0],
        second[1],
        angles="xy",
        scale_units="xy",
        scale=1.0,
        color="tab:blue",
        label="second primitive vector",
    )
    axis.set_aspect("equal")
    axis.set_title(title)
    axis.grid(True)
    axis.legend()

figure.tight_layout()
plt.show()

## Interpretation

Each reciprocal basis satisfies the defining duality relation to binary64 rounding. The direct and reciprocal fundamental-region measures multiply to $(2\pi)^d$ in dimension $d$. Integer coefficient vectors generate the plotted direct and reciprocal point sets through their respective basis matrices.

These physical vector bases are distinct from `FinitePeriodicDomain`, which represents only a finite quotient of integer indices and carries no physical vectors or units.

## Exercises

1. Replace the two-dimensional basis with a rectangular basis and derive its reciprocal vectors.
2. Change the orientation of the 3D basis while preserving its volume and verify the duality relation.
3. Use a batch of integer indices to construct selected direct and reciprocal vectors.
4. Reverse two 3D primitive vectors and examine the signed determinant separately from the positive fundamental-region measure.
